# 07 · scVI integration

*sc-rnaseq-wf, re-instantiated step by step for `team-jakobsson` (`jakobsson-pmdbs-sn-rnaseq`)*

| | |
| :- | :- |
| WDL task | [`cluster_data.integrate_sample_data`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/cohort_analysis/cluster_data/cluster_data.wdl) |
| Script | [`integrate_scvi`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/docker/sc_tools/scripts/main/integrate_scvi) → `sc-integrate-scvi` |

[scVI](https://docs.scvi-tools.org/en/stable/user_guide/models/scvi.html) is a variational autoencoder that models the raw counts (`layers["counts"]`, HVGs only) with a zero-inflated negative binomial likelihood, conditioned on the batch (`batch_id`). The 30-dimensional latent space `X_scVI` describes each cell's expression with the batch effect removed. It is used for clustering and UMAP (notebook 08), and the trained model initializes scANVI (notebook 09).

Fixed settings in the script: 2 hidden layers, 30 latent dimensions, gene-cell dispersion, ZINB likelihood, 85% train split, batch size 1024, early stopping (patience 20), and a maximum number of epochs from scvi-tools' heuristic for the number of cells (at least 20). Gradients are clipped at 5.0 to keep training stable. The learning rate is 1e-3, or 1e-4 above 3.05M cells. The script fails if the latent space contains NaN, which means training diverged.

**Requires a GPU** (`accelerator="gpu"`).

**Inputs**
- `${cohort_id}.mmc.h5ad`

**Outputs** (in `cfg.RUN_DIR`, named as in the WDL)
- `${cohort_id}.scvi_integrated.h5ad` (adds `obsm['X_scVI']`)
- `${cohort_id}_scvi_model/` (and `${cohort_id}.scvi_model.tar.gz`)

**Previous:** [06_transcriptional_phenotype](06_transcriptional_phenotype.ipynb) · **Next:** [08_clustering_umap](08_clustering_umap.ipynb)

In [ ]:
import wf_config as cfg
import pandas as pd
import scanpy as sc
from IPython.display import Image, display

cfg.RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run directory:", cfg.RUN_DIR)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

## 1. Run `sc-integrate-scvi`

In [ ]:
scvi_dir = cfg.RUN_DIR / f"{cfg.COHORT_ID}_scvi_model"
cfg.run(
    "sc-integrate-scvi",
    "--latent-key", cfg.SCVI_LATENT_KEY,
    "--batch-key", cfg.BATCH_KEY,
    "--adata-input", cfg.out("mmc.h5ad"),
    "--adata-output", cfg.out("scvi_integrated.h5ad"),
    "--output-scvi-dir", scvi_dir,
)

The WDL archives the model directory (the name must stay the same: scvi-tools expects `model.pt` inside it).

In [ ]:
import tarfile

with tarfile.open(cfg.out("scvi_model.tar.gz"), "w:gz") as tar:
    tar.add(scvi_dir, arcname=scvi_dir.name)

## 2. Inspect the model and latent space

In [ ]:
import numpy as np
import scvi

adata = sc.read_h5ad(cfg.out("scvi_integrated.h5ad"))
latent = adata.obsm[cfg.SCVI_LATENT_KEY]
print("latent shape:", latent.shape)
print("cells with NaN in the latent space:", int(np.isnan(latent).any(axis=1).sum()))

model = scvi.model.SCVI.load(scvi_dir, adata=adata)
history = pd.concat([model.history["elbo_train"], model.history["elbo_validation"]], axis=1)
history.plot(title="scVI ELBO", xlabel="epoch");

**Previous:** [06_transcriptional_phenotype](06_transcriptional_phenotype.ipynb) · **Next:** [08_clustering_umap](08_clustering_umap.ipynb)